In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from bound_propagation import BoundModelFactory, HyperRectangle
from tqdm import tqdm

In [2]:
# !pip3 install bound-propagation

In [3]:
device = torch.device("cuda" if torch.cuda.is_available()  else "cpu")
batch_size = 64

In [4]:
np.random.seed(42)
torch.manual_seed(42)

In [5]:
## Dataloaders - No normalization for IBP
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True,
                              transform=transforms.Compose([transforms.ToTensor()]))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True,
                             transform=transforms.Compose([transforms.ToTensor()]))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [6]:
class Net(nn.Module):
  def __init__(self, hidden_size=50):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(28*28, hidden_size)
    self.fc2 = nn.Linear(hidden_size, hidden_size)
    self.fc3 = nn.Linear(hidden_size, hidden_size)
    self.fc4 = nn.Linear(hidden_size, 10) # Output layer

  def forward(self, x):
    x = x.view((-1, 28*28))
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = self.fc4(x)
    return x

In [7]:
model = Net().to(device)
model.train()

Net(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=50, bias=True)
  (fc4): Linear(in_features=50, out_features=10, bias=True)
)

In [8]:
def compute_logits_worst_case(model, x, y, eps):
    n = x.size(0)
    flat_x = x.reshape(n, -1)

    # Define perturbed input range [x - eps, x + eps], clamped to valid pixel range
    lower = torch.clamp(flat_x - eps, 0.0, 1.0)
    upper = torch.clamp(flat_x + eps, 0.0, 1.0)
    bounds = HyperRectangle(lower, upper)

    # Reconstruct the sequential feedforward network
    layers = [
        model.fc1, nn.ReLU(),
        model.fc2, nn.ReLU(),
        model.fc3, nn.ReLU(),
        model.fc4
    ]
    sequential_net = nn.Sequential(*layers)

    # Initialize bounded model via factory
    bounded_model = BoundModelFactory().build(sequential_net)

    # Propagate bounds to compute logits interval
    logits_interval = bounded_model.ibp(bounds)
    lo, hi = logits_interval.lower, logits_interval.upper

    # Select lower bounds for true class, upper for all others
    idx = torch.arange(n)
    selector = torch.ones_like(lo, dtype=torch.bool)
    selector[idx, y] = False
    combined_logits = torch.where(selector, hi, lo)

    # Compute cross-entropy over worst-case logits
    loss = F.cross_entropy(combined_logits, y)
    return loss

### Part A

In [11]:
def train_ibp(model, train_loader, epochs=20, max_eps=0.1, device='cuda'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    log = {'loss': [], 'std_loss': [], 'rob_loss': []}

    print(f"Starting IBP training for {epochs} epochs (target ε={max_eps})")

    for ep in range(epochs):
        model.train()
        total_loss = total_std = total_rob = 0.0

        # Linear schedules
        kappa = 1.0 - 0.5 * (ep / epochs)
        eps = max_eps * (ep / epochs)

        for images, labels in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            # Standard CE
            logits = model(images)
            loss_std = F.cross_entropy(logits, labels)

            # Robust CE via IBP
            loss_rob = compute_logits_worst_case(model, images, labels, eps)

            # Combined objective
            loss = kappa * loss_std + (1 - kappa) * loss_rob
            loss.backward()
            optimizer.step()

        model.eval()
        tot_val, tot_acc = 0.0, 0.0
        val_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            with torch.no_grad():
                logits = model(images)
                ce_loss = nn.CrossEntropyLoss()(logits, labels)
                rb_loss = compute_logits_worst_case(model, images, labels, eps)
                loss = kappa * ce_loss + (1 - kappa) * rb_loss
                val_loss += loss.item()
                tot_acc += (logits.argmax(dim=1) == labels).sum().item()
                tot_val += labels.size(0)
        val_acc = 100.0 * tot_acc / tot_val

        print(f'Epoch {ep+1}/{epochs}, Loss: {val_loss/len(train_loader):.3f}, Accuracy: {val_acc:.2f}%')

        # print(f"Epoch {ep+1:2d}/{epochs} → Loss={avg_loss:.4f}, Std={avg_std:.4f}, Rob={avg_rob:.4f}")

    return log


# === Example usage ===
start = time.time()
history = train_ibp(model, train_loader, epochs=20, max_eps=0.1, device=device)
ibp_tt = time.time() - start

print(f"\nIBP Training completed in {ibp_tt:.2f}s ({ibp_tt/60:.2f}m)")

Starting IBP training for 20 epochs (target ε=0.1)


Epoch 1/20: 100%|██████████| 938/938 [00:04<00:00, 224.77it/s]


Epoch 1/20, Loss: 0.212, Accuracy: 93.84%


Epoch 2/20:  89%|████████▉ | 838/938 [00:03<00:00, 244.05it/s]


KeyboardInterrupt: 

In [10]:
def train_standard(model, train_loader, epochs=5, lr=1e-3, device='cuda'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    start = time.time()

    for ep in range(epochs):
        model.train()
        total_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"Epoch {ep+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            loss = F.cross_entropy(model(images), labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f'Epoch {ep+1}/{epochs}, Loss: {total_loss/len(train_loader):.3f}')

    return time.time() - start


# === Example usage and timing comparison ===
std_model = Net().to(device)
standard_tt = train_standard(std_model, train_loader, epochs=20, lr=1e-3, device=device)

diff = ibp_tt - standard_tt
print(f"\nTraining Time Comparison:")
print(f"  IBP: {ibp_tt:.2f}s ({ibp_tt/60:.2f}m)")
print(f"  Standard: {standard_tt:.2f}s ({standard_tt/60:.2f}m)")


Epoch 1/20: 100%|██████████| 938/938 [00:02<00:00, 423.14it/s]


Epoch 1/20, Loss: 0.435


Epoch 2/20: 100%|██████████| 938/938 [00:02<00:00, 439.54it/s]


Epoch 2/20, Loss: 0.178


Epoch 3/20: 100%|██████████| 938/938 [00:02<00:00, 442.92it/s]


...

Epoch 19/20: 100%|██████████| 938/938 [00:02<00:00, 434.11it/s]


Epoch 19/20, Loss: 0.019


Epoch 20/20: 100%|██████████| 938/938 [00:02<00:00, 436.97it/s]

Epoch 20/20, Loss: 0.017


NameError: name 'ibp_tt' is not defined

In [10]:
def pgd_attack(model, x, y, eps: float = 0.1, step_size: float = 0.01, steps: int = 40):
    model.eval()
    batch_size = x.size(0)

    # Flatten input for consistent gradient operations
    x_orig = x.view(batch_size, -1)
    x_adv = x_orig.clone().detach()

    for _ in range(steps):
        x_adv.requires_grad_(True)

        logits = model(x_adv.view_as(x))
        loss = F.cross_entropy(logits, y)

        # Compute gradients w.r.t input
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + step_size * grad.sign()

        # Project perturbation back to ε-ball
        delta = torch.clamp(x_adv - x_orig, min=-eps, max=eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0)

    # Reshape back to original input dimensions
    return x_adv.view_as(x)


In [13]:
def evaluate_accuracy_and_robustness(model, loader, eps=0.1, device="cuda"):
    model.eval()
    correct_clean = correct_adv = total = 0

    for x, y in tqdm(loader, desc="Evaluating"):
        x, y = x.to(device), y.to(device)
        total += y.size(0)

        with torch.no_grad():
            clean_pred = model(x).argmax(1)
            correct_clean += (clean_pred == y).sum().item()

        x_adv = pgd_attack(model, x, y, eps=eps, step_size=0.01, steps=40)
        with torch.no_grad():
            adv_pred = model(x_adv).argmax(1)
            correct_adv += (adv_pred == y).sum().item()

    std_acc = 100 * correct_clean / total
    rob_acc = 100 * correct_adv / total
    return std_acc, rob_acc


# === Evaluate both models ===
ibp_std_acc, ibp_rob_acc = evaluate_accuracy_and_robustness(model, test_loader, eps=0.1, device=device)
std_std_acc, std_rob_acc = evaluate_accuracy_and_robustness(std_model, test_loader, eps=0.1, device=device)

print("\nResults Summary")
print("-" * 50)
print(f"IBP Model     - Standard: {ibp_std_acc:.2f}% | Robust: {ibp_rob_acc:.2f}%")
print(f"Standard Model- Standard: {std_std_acc:.2f}% | Robust: {std_rob_acc:.2f}%")

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 56.55it/s]


Results Summary
--------------------------------------------------
IBP Model     - Standard: 95.24% | Robust: 83.81%
Standard Model- Standard: 97.22% | Robust: 1.03%


### Part B

In [ ]:
def verify_robustness_single_image(model, x, y, eps):
    x_flat = x.view(1, -1)
    lower_bound = torch.clamp(x_flat - eps, 0.0, 1.0)
    upper_bound = torch.clamp(x_flat + eps, 0.0, 1.0)
    input_bounds = HyperRectangle(lower_bound, upper_bound)

    # Build sequential model for IBP propagation
    layers = [
        model.fc1, nn.ReLU(),
        model.fc2, nn.ReLU(),
        model.fc3, nn.ReLU(),
        model.fc4
    ]
    bound_model = nn.Sequential(*layers)

    # Construct bounded model and compute output bounds
    bounded_net = BoundModelFactory().build(bound_model)
    bounds = bounded_net.ibp(input_bounds)

    lower, upper = bounds.lower[0], bounds.upper[0]
    true_lower = lower[y]

    # Verify that all other classes’ upper bounds stay below the true label’s lower bound
    return not any(upper[i] >= true_lower for i in range(len(lower)) if i != y)

In [ ]:
def compute_verified_accuracy(model, loader, eps, device="cuda"):
    model.eval()
    verified = correct = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            preds = model(x).argmax(dim=1)

        for i in range(len(x)):
            if preds[i].item() == y[i].item():
                correct += 1
                if verify_robustness_single_image(model, x[i:i+1], y[i].item(), eps):
                    verified += 1

    acc = 100.0 * verified / correct if correct > 0 else 0.0
    return verified, correct, acc

In [ ]:
epsilon_values = np.linspace(0.01, 0.1, 10)
verified_results = []

print("Verified Accuracy Evaluation (Box Verification)")

for eps in epsilon_values:
    num_verified, num_correct, verified_acc = compute_verified_accuracy(model, test_loader, eps, device=device)
    verified_results.append((eps, num_verified, num_correct, verified_acc))
    print(f"ε={eps:.3f} → Verified: {num_verified}/{num_correct} ({verified_acc:.2f}%)")

Verified Accuracy Evaluation (Box Verification)
ε=0.010 → Verified: 9519/9602 (99.14%)
ε=0.020 → Verified: 9419/9602 (98.09%)
ε=0.030 → Verified: 9323/9602 (97.09%)
ε=0.040 → Verified: 9196/9602 (95.77%)
ε=0.050 → Verified: 9041/9602 (94.16%)
ε=0.060 → Verified: 8877/9602 (92.45%)
ε=0.070 → Verified: 8664/9602 (90.23%)
ε=0.080 → Verified: 8419/9602 (87.68%)
ε=0.090 → Verified: 8164/9602 (85.02%)
ε=0.100 → Verified: 7764/9602 (80.86%)
